# RNA Stability Design Bridge

Proof-of-concept notebook for a modular RNA design augmentation layer.

This notebook is intentionally standalone. It demonstrates how sequence features and a stability model can be connected to gradient-based sequence design.

**Data**: this notebook loads a real dataset when available. Place a CSV at `DATA_PATH` (default `data/rna_stability.csv`) with at least:

- a sequence column (default name `sequence`, DNA or RNA, any case)
- a numeric stability/degradation label column (auto-detected from `stability`, `half_life`, `reactivity`, `deg_Mg_pH10`, `deg_Mg_50C` if not given explicitly)

If no such file is found, the notebook falls back to a **synthetic mock** dataset (random sequences scored by a hand-written `stability_proxy` heuristic) so the pipeline still runs end-to-end for demonstration. Synthetic-mode results are clearly labeled and should not be interpreted as biology.

The goal is not to replace existing RNA analytics workflows. The goal is to show a lightweight extension layer that connects sequence features (real or mock) to a stability model and a gradient-based sequence optimizer, with held-out evaluation (R², Spearman correlation) to check the model actually generalizes.

In [ ]:
# Optional dependency installation
# If needed, uncomment the line below.
# %pip install -q numpy pandas matplotlib torch jax jaxlib optax

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import jax
    import jax.numpy as jnp
    import optax
    HAS_JAX = True
except Exception as e:
    HAS_JAX = False
    print('JAX not available:', e)

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print('PyTorch not available:', e)

In [ ]:
BASES = ['A', 'C', 'G', 'U']
BASE_TO_IDX = {base: idx for idx, base in enumerate(BASES)}


def sanitize_sequence(seq):
    return seq.upper().replace('T', 'U')


def one_hot_encode(seq, length=None):
    seq = sanitize_sequence(seq)
    if length is None:
        length = len(seq)

    arr = np.zeros((length, len(BASES)), dtype=np.float32)

    for i, base in enumerate(seq[:length]):
        idx = BASE_TO_IDX.get(base)
        if idx is not None:
            arr[i, idx] = 1.0

    return arr


def gc_content(seq):
    seq = sanitize_sequence(seq)
    if len(seq) == 0:
        return 0.0
    return (seq.count('G') + seq.count('C')) / len(seq)


def homopolymer_penalty(seq, max_run=4):
    seq = sanitize_sequence(seq)
    if len(seq) == 0:
        return 0.0

    penalty = 0
    run_length = 1
    previous = seq[0]

    for base in seq[1:]:
        if base == previous:
            run_length += 1
        else:
            if run_length > max_run:
                penalty += run_length - max_run
            run_length = 1
            previous = base

    if run_length > max_run:
        penalty += run_length - max_run

    return min(1.0, penalty / len(seq))


def stability_proxy(seq):
    seq = sanitize_sequence(seq)

    if len(seq) == 0:
        return 0.0

    gc = gc_content(seq)

    gc_term = np.exp(-((gc - 0.55) ** 2) / (2 * 0.12 ** 2))
    hp_term = 1.0 - homopolymer_penalty(seq)

    uuu_density = seq.count('UUU') / max(1, len(seq) - 2)
    uuu_term = 1.0 - min(1.0, uuu_density * 10.0)

    score = (
        0.5 * gc_term
        + 0.3 * hp_term
        + 0.2 * uuu_term
    )

    return float(np.clip(score, 0.0, 1.0))


def make_dataset(sequences, length=80):
    """Synthetic demo path: builds X, y from raw sequences using stability_proxy as a mock label."""
    X = []
    y = []
    clean_sequences = []

    for item in sequences:
        if isinstance(item, tuple):
            _, seq = item
        else:
            seq = item

        seq = sanitize_sequence(seq)

        if len(seq) < 20:
            continue

        seq = seq[:length]

        X.append(one_hot_encode(seq, length=length).reshape(-1))
        y.append(stability_proxy(seq))
        clean_sequences.append(seq)

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    return X, y, clean_sequences


def load_real_dataset(path, seq_col='sequence', label_col=None):
    """Load a real RNA stability dataset from CSV.

    Expects at least a sequence column and a numeric label column. If
    label_col is None, tries common column names in order.
    """
    df = pd.read_csv(path)

    if seq_col not in df.columns:
        raise ValueError(f"Column '{seq_col}' not found in {path}. Available columns: {list(df.columns)}")

    if label_col is None:
        candidates = ['stability', 'half_life', 'reactivity', 'deg_Mg_pH10', 'deg_Mg_50C']
        label_col = next((c for c in candidates if c in df.columns), None)

        if label_col is None:
            raise ValueError(
                f"Could not auto-detect a label column in {path}. "
                f"Available columns: {list(df.columns)}. Pass label_col explicitly."
            )

    df = df[[seq_col, label_col]].dropna()
    df.attrs['label_col'] = label_col

    return df


def make_dataset_from_dataframe(df, seq_col='sequence', label_col='stability', length=80):
    """Real-data path: builds X, y from a dataframe using measured labels (no mock proxy)."""
    X = []
    y = []
    clean_sequences = []

    for seq, label in zip(df[seq_col], df[label_col]):
        seq = sanitize_sequence(str(seq))

        if len(seq) < 20:
            continue

        seq = seq[:length]

        X.append(one_hot_encode(seq, length=length).reshape(-1))
        y.append(float(label))
        clean_sequences.append(seq)

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    return X, y, clean_sequences

In [ ]:
import random


def random_sequence(length=80, gc_target=0.5):
    """Synthetic sequence generator, used only when no real dataset is found."""
    seq = []

    for _ in range(length):
        if random.random() < gc_target:
            seq.append(random.choice('CG'))
        else:
            seq.append(random.choice('AU'))

    random.shuffle(seq)
    return ''.join(seq)

In [ ]:
import os

DATA_PATH = 'data/rna_stability.csv'  # point this at your own sequence/stability CSV
SEQ_COL = 'sequence'
LABEL_COL = None  # auto-detected if None; see load_real_dataset for candidate names

if os.path.exists(DATA_PATH):
    df_real = load_real_dataset(DATA_PATH, seq_col=SEQ_COL, label_col=LABEL_COL)
    X, y, clean_sequences = make_dataset_from_dataframe(
        df_real, seq_col=SEQ_COL, label_col=df_real.attrs['label_col'], length=80
    )
    data_source = f"real ({DATA_PATH}, label='{df_real.attrs['label_col']}')"
else:
    print(f'No real dataset found at {DATA_PATH}.')
    print('Falling back to SYNTHETIC mock data for demonstration only --')
    print('these are NOT real measurements. Replace DATA_PATH with a real')
    print('sequence/stability CSV before drawing biological conclusions.')

    random.seed(42)
    sequences = [
        random_sequence(length=80, gc_target=random.uniform(0.35, 0.65))
        for _ in range(300)
    ]
    X, y, clean_sequences = make_dataset(sequences, length=80)
    data_source = 'synthetic (mock stability_proxy)'

y_min, y_max = float(y.min()), float(y.max())
y_range = y_max - y_min if y_max > y_min else 1.0
y = (y - y_min) / y_range

print('Data source:', data_source)
print('X shape:', X.shape)
print('y shape:', y.shape)
print(f'Labels normalized to [0, 1] from original range [{y_min:.4f}, {y_max:.4f}]')

# Train / validation / test split (70 / 15 / 15), fixed seed for reproducibility
rng = np.random.RandomState(42)
perm = rng.permutation(len(X))

n_train = int(0.7 * len(X))
n_val = int(0.15 * len(X))

train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(y, bins=30, color='steelblue', edgecolor='black')
plt.xlabel('Stability label (normalized 0-1)')
plt.ylabel('Count')
plt.title(f'Distribution of stability labels -- {data_source}')
plt.tight_layout()
plt.show()

In [ ]:
if HAS_JAX:
    def init_params(key, input_dim, hidden_dim=64):
        k1, k2 = jax.random.split(key)
        w1 = jax.random.normal(k1, (input_dim, hidden_dim)) * 0.01
        b1 = jnp.zeros(hidden_dim)
        w2 = jax.random.normal(k2, (hidden_dim, 1)) * 0.01
        b2 = jnp.zeros(1)
        return {'w1': w1, 'b1': b1, 'w2': w2, 'b2': b2}

    def forward(params, x):
        h = jax.nn.relu(x @ params['w1'] + params['b1'])
        y = jax.nn.sigmoid(h @ params['w2'] + params['b2'])
        return y.squeeze(-1)

    def loss_fn(params, x, y):
        pred = forward(params, x)
        return jnp.mean((pred - y) ** 2)

    def train_jax(X_train, y_train, X_val, y_val, epochs=200, lr=0.01, seed=42):
        X_train = jnp.asarray(X_train, dtype=jnp.float32)
        y_train = jnp.asarray(y_train, dtype=jnp.float32)
        X_val = jnp.asarray(X_val, dtype=jnp.float32)
        y_val = jnp.asarray(y_val, dtype=jnp.float32)

        key = jax.random.PRNGKey(seed)
        params = init_params(key, input_dim=X_train.shape[1])

        optimizer = optax.adam(lr)
        opt_state = optimizer.init(params)

        @jax.jit
        def step(params, opt_state, x, y):
            loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
            updates, opt_state = optimizer.update(grads, opt_state, params)
            params = optax.apply_updates(params, updates)
            return params, opt_state, loss

        history = []
        for epoch in range(epochs):
            params, opt_state, train_loss = step(params, opt_state, X_train, y_train)
            val_loss = float(loss_fn(params, X_val, y_val))
            history.append((float(train_loss), val_loss))

        return params, history

    def design_jax(params, length, target_gc=0.55, steps=300, lr=0.1, seed=42):
        key = jax.random.PRNGKey(seed)
        logits = jax.random.normal(key, (length, 4)) * 0.1

        optimizer = optax.adam(lr)
        opt_state = optimizer.init(logits)

        def design_loss(logits):
            probs = jax.nn.softmax(logits, axis=-1)
            flat = probs.reshape(-1)
            score = forward(params, flat[None, :])[0]

            gc = (probs[:, 1] + probs[:, 2]).mean()
            gc_penalty = 10.0 * (gc - target_gc) ** 2

            entropy = -jnp.sum(
                probs * jnp.log(probs + 1e-8),
                axis=-1
            ).mean()

            loss = -score + gc_penalty + 0.05 * entropy
            return loss, (score, gc, entropy)

        @jax.jit
        def step(logits, opt_state):
            (loss, metrics), grads = jax.value_and_grad(
                design_loss,
                has_aux=True
            )(logits)

            updates, opt_state = optimizer.update(grads, opt_state, logits)
            logits = optax.apply_updates(logits, updates)

            return logits, opt_state, loss, metrics

        history = []
        for _ in range(steps):
            logits, opt_state, loss, metrics = step(logits, opt_state)
            history.append(
                (
                    float(loss),
                    float(metrics[0]),
                    float(metrics[1]),
                    float(metrics[2])
                )
            )

        probs = np.asarray(jax.nn.softmax(logits, axis=-1))
        seq_indices = np.argmax(probs, axis=-1)
        bases = np.array(['A', 'C', 'G', 'U'])
        seq = ''.join(bases[i] for i in seq_indices)

        return seq, probs, history

else:
    print('Skipping JAX model definitions because JAX is not available.')

In [ ]:
jax_params = None
jax_history = []

if HAS_JAX:
    jax_params, jax_history = train_jax(
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=200,
        lr=0.01,
        seed=42
    )
    final_train_loss, final_val_loss = jax_history[-1]
    print('Final JAX training loss:', final_train_loss)
    print('Final JAX validation loss:', final_val_loss)
else:
    print('JAX training skipped.')

In [ ]:
jax_seq = None
jax_probs = None
jax_design_history = []

if HAS_JAX and jax_params is not None:
    jax_seq, jax_probs, jax_design_history = design_jax(
        params=jax_params,
        length=80,
        target_gc=0.55,
        steps=300,
        lr=0.1,
        seed=42
    )

    print('JAX-designed sequence:')
    print(jax_seq)
else:
    print('JAX design skipped.')

In [ ]:
if len(jax_design_history) > 0:
    jax_design_df = pd.DataFrame(
        jax_design_history,
        columns=['loss', 'score', 'gc', 'entropy']
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].plot(jax_design_df['score'])
    axes[0].set_title('Predicted stability score')

    axes[1].plot(jax_design_df['gc'])
    axes[1].set_title('GC fraction')

    axes[2].plot(jax_design_df['entropy'])
    axes[2].set_title('Sequence entropy')

    for ax in axes:
        ax.set_xlabel('Optimization step')

    plt.tight_layout()
    plt.show()

else:
    print('No JAX design history available.')

In [ ]:
if HAS_JAX and jax_params is not None:
    test_preds = np.asarray(forward(jax_params, jnp.asarray(X_test, dtype=jnp.float32)))

    ss_res = float(np.sum((y_test - test_preds) ** 2))
    ss_tot = float(np.sum((y_test - y_test.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')

    spearman = pd.Series(test_preds).rank().corr(pd.Series(y_test).rank())

    print(f'JAX held-out test set (n = {len(y_test)}):')
    print(f'  R^2:      {r2:.4f}')
    print(f'  Spearman: {spearman:.4f}')

    plt.figure(figsize=(5, 5))
    plt.scatter(y_test, test_preds, alpha=0.6, edgecolor='black')
    lims = [min(y_test.min(), test_preds.min()), max(y_test.max(), test_preds.max())]
    plt.plot(lims, lims, 'r--', linewidth=1)
    plt.xlabel('Actual label (normalized)')
    plt.ylabel('Predicted label')
    plt.title('JAX model: held-out test predictions')
    plt.tight_layout()
    plt.show()
else:
    print('JAX held-out evaluation skipped.')

In [ ]:
if HAS_TORCH:
    class StabilityMLP(nn.Module):
        def __init__(self, input_dim, hidden_dim=64):
            super().__init__()

            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
                nn.Sigmoid()
            )

        def forward(self, x):
            return self.net(x).squeeze(-1)

    def train_torch(X_train, y_train, X_val, y_val, epochs=200, lr=0.01, device='cpu'):
        torch.manual_seed(42)

        X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
        X_val_t = torch.tensor(X_val, dtype=torch.float32, device=device)
        y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)

        model = StabilityMLP(input_dim=X_train.shape[1]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()

        history = []

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()

            pred = model(X_train_t)
            loss = loss_fn(pred, y_train_t)

            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                val_loss = loss_fn(model(X_val_t), y_val_t)

            history.append((float(loss), float(val_loss)))

        model.eval()

        return model, history

    def design_torch(model, length, target_gc=0.55, steps=300, lr=0.1, device='cpu'):
        model.eval()

        torch.manual_seed(42)

        logits = (
            torch.randn(length, 4, device=device) * 0.1
        ).requires_grad_(True)

        optimizer = torch.optim.Adam([logits], lr=lr)

        history = []

        for step in range(steps):
            optimizer.zero_grad()

            probs = torch.softmax(logits, dim=-1)
            flat = probs.reshape(1, -1)

            score = model(flat)[0]

            gc = probs[:, [1, 2]].sum(dim=-1).mean()
            gc_penalty = 10.0 * (gc - target_gc) ** 2

            entropy = (
                -probs * torch.log(probs + 1e-8)
            ).sum(dim=-1).mean()

            loss = -score + gc_penalty + 0.05 * entropy

            loss.backward()
            optimizer.step()

            history.append(
                (
                    float(loss),
                    float(score),
                    float(gc),
                    float(entropy)
                )
            )

        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        seq_indices = probs.argmax(axis=-1)
        bases = np.array(['A', 'C', 'G', 'U'])
        seq = ''.join(bases[i] for i in seq_indices)

        return seq, probs, history

else:
    print('Skipping PyTorch model definitions because PyTorch is not available.')

In [ ]:
torch_model = None
torch_history = []

if HAS_TORCH:
    torch_model, torch_history = train_torch(
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=200,
        lr=0.01,
        device='cpu'
    )

    final_train_loss, final_val_loss = torch_history[-1]
    print('Final PyTorch training loss:', final_train_loss)
    print('Final PyTorch validation loss:', final_val_loss)
else:
    print('PyTorch training skipped.')

In [ ]:
torch_seq = None
torch_probs = None
torch_design_history = []

if HAS_TORCH and torch_model is not None:
    torch_seq, torch_probs, torch_design_history = design_torch(
        model=torch_model,
        length=80,
        target_gc=0.55,
        steps=300,
        lr=0.1,
        device='cpu'
    )

    print('PyTorch-designed sequence:')
    print(torch_seq)
else:
    print('PyTorch design skipped.')

In [ ]:
if len(torch_design_history) > 0:
    torch_design_df = pd.DataFrame(
        torch_design_history,
        columns=['loss', 'score', 'gc', 'entropy']
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].plot(torch_design_df['score'])
    axes[0].set_title('Predicted stability score')

    axes[1].plot(torch_design_df['gc'])
    axes[1].set_title('GC fraction')

    axes[2].plot(torch_design_df['entropy'])
    axes[2].set_title('Sequence entropy')

    for ax in axes:
        ax.set_xlabel('Optimization step')

    plt.tight_layout()
    plt.show()

else:
    print('No PyTorch design history available.')

In [ ]:
if HAS_TORCH and torch_model is not None:
    with torch.no_grad():
        test_preds = torch_model(torch.tensor(X_test, dtype=torch.float32)).cpu().numpy()

    ss_res = float(np.sum((y_test - test_preds) ** 2))
    ss_tot = float(np.sum((y_test - y_test.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')

    spearman = pd.Series(test_preds).rank().corr(pd.Series(y_test).rank())

    print(f'PyTorch held-out test set (n = {len(y_test)}):')
    print(f'  R^2:      {r2:.4f}')
    print(f'  Spearman: {spearman:.4f}')

    plt.figure(figsize=(5, 5))
    plt.scatter(y_test, test_preds, alpha=0.6, edgecolor='black')
    lims = [min(y_test.min(), test_preds.min()), max(y_test.max(), test_preds.max())]
    plt.plot(lims, lims, 'r--', linewidth=1)
    plt.xlabel('Actual label (normalized)')
    plt.ylabel('Predicted label')
    plt.title('PyTorch model: held-out test predictions')
    plt.tight_layout()
    plt.show()
else:
    print('PyTorch held-out evaluation skipped.')

In [ ]:
print('JAX sequence available:', jax_seq is not None)
print('PyTorch sequence available:', torch_seq is not None)

if jax_seq is not None:
    print('JAX sequence GC:', gc_content(jax_seq))

if torch_seq is not None:
    print('PyTorch sequence GC:', gc_content(torch_seq))

# Summary and extension path

This notebook demonstrates a minimal RNA design augmentation layer:

1. RNA sequences are loaded from a real dataset when available (`DATA_PATH`), or generated synthetically as a fallback demo.
2. Labels are normalized to [0, 1] and split into train / validation / test sets (70 / 15 / 15).
3. A small machine learning model (JAX and PyTorch implementations) learns to predict the stability label, with validation loss tracked during training.
4. Model generalization is checked on the held-out test set via R² and Spearman correlation, plus a predicted-vs-actual scatter plot.
5. Gradient-based optimization proposes sequences with improved predicted stability, subject to a GC-content target and an entropy regularizer.

To use this with real data, drop a CSV at `DATA_PATH` with a `sequence` column and a numeric stability/degradation column (e.g. measured half-life, reactivity, or degradation rate from RNA-seq decay time-courses, direct RNA sequencing, or in-vitro stability assays). No other code changes are required — `load_real_dataset` auto-detects common label column names.

Further extensions worth considering:

- Replace the entropy/GC-only design constraints with real biophysical checks (e.g. RNA secondary-structure folding energy via ViennaRNA/RNAfold).
- Collapse the JAX and PyTorch tracks into a single implementation once one is chosen for production use.
- Score optimizer-proposed sequences against an independent model or dataset (not the one used for training) before trusting the "improved stability" claim.